# Phase 4 模型訓練測試（Regime-specific + Multi-timeframe）

- 驗證 50 維跨市場特徵
- 載入 all_day / settlement 模型 smoke test
- 讀取 model_comparison_summary.md
- regime 樣本數檢查


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from features.feature_utils import FEATURES_PER_SYMBOL
from models.feature_preparation import prepare_cross_market_features
from models.model_utils import Phase4Paths, load_model
from models.regime_utils import ALL_REGIMES, filter_by_regime
from utils.config import SYMBOLS

p4 = Phase4Paths.default()
print('Models root =', p4.models_root())
print('FEATURES_PER_SYMBOL =', FEATURES_PER_SYMBOL)


In [ ]:
for sym in SYMBOLS:
    prep = prepare_cross_market_features(sym, regime='all_day')
    print(sym, 'all_day X shape', prep.X.shape, 'expected features', FEATURES_PER_SYMBOL * 2)
    for reg in ALL_REGIMES:
        try:
            sub = prepare_cross_market_features(sym, regime=reg)
            print(f'  {reg}: n={len(sub.X)}')
        except ValueError as e:
            print(f'  {reg}: skip ({e})')


In [ ]:
summary_path = p4.comparison_summary_path()
if summary_path.is_file():
    print(summary_path.read_text(encoding='utf-8')[:2000])
else:
    print('尚未產生 comparison summary，請先執行 model_pipeline.py')


In [ ]:
sym = SYMBOLS[0]
label_type = 'label_realized_volatility'
for regime in ('all_day', 'settlement'):
    try:
        m = load_model(sym, label_type, regime=regime)
        prep = prepare_cross_market_features(sym, regime=regime)
        X = prep.X.tail(500)
        proba = m.predict_proba(X)
        print(f'OK predict {sym} {regime}: proba shape', proba.shape)
    except FileNotFoundError as e:
        print(f'skip {regime}:', e)
